**Loading the Dataset**

In [ ]:
#Import libraries
import sys
print(sys.executable)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow_hub as hub
import textattack
from textattack.models.wrappers import ModelWrapper
# Load the training data

#Replace the path above with the folder where YOUR file is saved
#Example: train_data = pd.read_csv("C:/Users/YourName/Documents/train.csv")
df = pd.read_csv("C:\\Users\\pande\\archive\\spam.csv", encoding='latin-1')
df.head()


**Basic Data Cleaning (Preprocessing)**

In [ ]:
df = df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)
df = df.rename(columns={'v1': 'label', 'v2': 'Text'})
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

**Splitting the Data**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['Text'],
    df['label_enc'],
    test_size=0.2,
    random_state=42
)

X_train_np = X_train.to_numpy()
X_test_np = X_test.to_numpy()
y_train_np = y_train.to_numpy()
y_test_np = y_test.to_numpy()

In [ ]:
avg_words_len = round(sum([len(i.split())
                      for i in df['Text']]) / len(df['Text']))
total_words_length = len(set(" ".join(df['Text']).split()))

print(f"Data Loaded. Training samples: {len(X_train_np)}")
print(f"Average words per message: {avg_words_len}")
print(f"Approximate vocabulary size: {total_words_length}")

In [ ]:
def compile_and_fit(model, epochs=5):
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    history = model.fit(
        X_train_np,
        y_train_np,
        epochs=epochs,
        validation_data=(X_test_np, y_test_np)
    )
    return history

def get_metrics(model, X, y):
    y_preds = np.round(model.predict(X))
    return {
        'accuracy': accuracy_score(y, y_preds),
        'precision': precision_score(y, y_preds),
        'recall': recall_score(y, y_preds),
        'f1-score': f1_score(y, y_preds)
    }

In [ ]:
from tensorflow.keras.layers import TextVectorization
text_vec = TextVectorization(
    max_tokens=total_words_length,
    standardize='lower_and_strip_punctuation',
    output_mode='int',
    output_sequence_length=avg_words_len
)
text_vec.adapt(X_train_np)

**Training a Basic Classification Model**

Defining Dense Embedding Model

In [ ]:

input_layer = layers.Input(shape=(1,), dtype=tf.string)
x = text_vec(input_layer)
x = layers.Embedding(input_dim=total_words_length, output_dim=128)(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(32, activation='relu')(x)
output_layer = layers.Dense(1, activation='sigmoid')(x)

model_1 = keras.Model(input_layer, output_layer, name="Dense_Model")
history_1 = compile_and_fit(model_1)

Defining Bi - LSTM Model

In [ ]:
input_layer = layers.Input(shape=(1,), dtype=tf.string)
x = text_vec(input_layer)
x = layers.Embedding(input_dim=total_words_length, output_dim=128)(x)
x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
x = layers.Flatten()(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(32, activation='relu')(x)
output_layer = layers.Dense(1, activation='sigmoid')(x)

model_2 = keras.Model(input_layer, output_layer, name="BiLSTM_Model")
history_2 = compile_and_fit(model_2)

Defining USE Model

In [ ]:
use_layer = hub.KerasLayer(
    "https://tfhub.dev/google/universal-sentence-encoder/4",
    trainable=False,
    input_shape=[],
    dtype=tf.string,
    name='USE'
)
input_layer = layers.Input(shape=[], dtype=tf.string)
embedding = layers.Lambda(lambda x: use_layer(
    x), output_shape=(512,))(input_layer)
x = layers.Dense(64, activation='relu')(embedding)
x = layers.Dropout(0.2)(x)
output_layer = layers.Dense(1, activation='sigmoid')(x)
model_3 = keras.Model(input_layer, output_layer, name="USE_Model")
history_3 = compile_and_fit(model_3)

Getting Results

In [ ]:
results = {
    'Dense Embedding': get_metrics(model_1, X_test_np, y_test_np),
    'Bi-LSTM': get_metrics(model_2, X_test_np, y_test_np),
    'Transfer Learning (USE)': get_metrics(model_3, X_test_np, y_test_np)
}

results_df = pd.DataFrame(results).transpose()
print("Performance Table:")
print(results_df)

Plotting Results

In [ ]:
results_df.plot(kind='bar', figsize=(10, 6))
plt.title("Model Performance Metrics (Bar Chart)")
plt.ylabel("Score")
plt.ylim(0.8, 1.0)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

for model_name in results_df.index:
    plt.plot(
        results_df.columns,
        results_df.loc[model_name],
        marker='o',
        label=model_name,
        linewidth=2
    )
plt.title("Model Performance Trends (Line Graph)")
plt.ylabel("Score")
plt.xlabel("Metric")
plt.ylim(0.8, 1.0)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

Creating Class Wrapper

In [ ]:
#defining wrapper class to use textattack
class CustomModelWrapper:
    def __init__(self,model,):
        self.model=model

    def __call__(self,text):
        inputs = np.array(text).flatten()
        
        # 2. Convert to a TensorFlow String Tensor
        # This is the critical step to fix the 'Invalid dtype' error
        tf_inputs = tf.constant(inputs, dtype=tf.string)
        
        # 3. Get predictions (verbose=0 keeps the console clean during attacks)
        preds = self.model.predict(tf_inputs, verbose=0)
        if preds.shape[1] == 1:
            return np.array([[1-p[0], p[0]] for p in preds])
        return preds
#creating wrapper objects
embed_wrapper=CustomModelWrapper(model_1)
lstm_wrapper=CustomModelWrapper(model_2)
use_wrapper=CustomModelWrapper(model_3)


Creating Attack Samples - Bi - LSTM

In [ ]:
from textattack.attack_recipes import TextBuggerLi2018
from textattack.attacker import Attacker,AttackArgs
from textattack.datasets import Dataset

lstm_attack = TextBuggerLi2018.build(lstm_wrapper)
samples = list(zip(X_test,y_test))

# Convert to the TextAttack-specific Dataset format
attack_dataset = Dataset(samples)
attack_args = AttackArgs(
    num_examples=300, 
    log_to_csv="textbugger_results_lstm.csv"
)

attacker = Attacker(lstm_attack, attack_dataset, attack_args)
results = attacker.attack_dataset()

Creating Attack Samples - USE

In [ ]:
use_attack = TextBuggerLi2018.build(use_wrapper)
samples = list(zip(X_test,y_test))

# Convert to the TextAttack-specific Dataset format
attack_dataset = Dataset(samples)
attack_args = AttackArgs(
    num_examples=300, 
    log_to_csv="textbugger_results_use.csv"
)

attacker = Attacker(use_attack, attack_dataset, attack_args)
results = attacker.attack_dataset()

Creating Attack Samples - Dense Embedding

In [ ]:
embed_attack = TextBuggerLi2018.build(embed_wrapper)
samples = list(zip(X_test,y_test))

# Convert to the TextAttack-specific Dataset format
attack_dataset = Dataset(samples)
attack_args = AttackArgs(
    num_examples=300, 
    log_to_csv="textbugger_results_embed.csv"
)

attacker = Attacker(embed_attack, attack_dataset, attack_args)
results = attacker.attack_dataset()

Adding Successful Attacks to Training Data

In [ ]:
lstm_results_df=pd.read_csv("textbugger_results_lstm.csv")
use_results_df=pd.read_csv("textbugger_results_use.csv")
embed_results_df=pd.read_csv("textbugger_results_embed.csv")

successful_lstm_rows=lstm_results_df[lstm_results_df['result_type']=='Successful']
successful_lstm_texts=successful_lstm_rows['perturbed_text'].values

successful_use_rows=use_results_df[use_results_df['result_type']=='Successful']
successful_use_texts=successful_use_rows['perturbed_text'].values

successful_embed_rows=embed_results_df[embed_results_df['result_type']=='Successful']
successful_embed_texts=successful_embed_rows['perturbed_text'].values


successful_text_label=np.ones(len(successful_lstm_texts)+len(successful_use_texts)+len(successful_embed_texts))
improved_X_train=np.concatenate([X_train_np,successful_lstm_texts,successful_use_texts,successful_embed_texts ])
improved_y_train=np.concatenate([y_train_np,successful_text_label])

print("Old training size:",len(X_train_np))
print("New training size:",len(improved_X_train) )


Getting Results with New Training Data

In [ ]:
history_1_improved=compile_and_fit(model_1,epochs=3)
history_2_improved=compile_and_fit(model_2,epochs=3)
history_3_improved=compile_and_fit(model_3,epochs=3)

results = {
    'Dense Embedding': get_metrics(model_1, X_test_np, y_test_np),
    'Bi-LSTM': get_metrics(model_2, X_test_np, y_test_np),
    'Transfer Learning (USE)': get_metrics(model_3, X_test_np, y_test_np)
}

Second Round of Attacks - Bi - LSTM

In [ ]:
lstm_attack = TextBuggerLi2018.build(lstm_wrapper)
samples = list(zip(X_test,y_test))

# Convert to the TextAttack-specific Dataset format
attack_dataset = Dataset(samples)
attack_args = AttackArgs(
    num_examples=100, 
    log_to_csv="textbugger_results_second_try_lstm.csv"
)

attacker = Attacker(lstm_attack, attack_dataset, attack_args)
results = attacker.attack_dataset()

Second Round of Attacks - USE

In [ ]:
use_attack = TextBuggerLi2018.build(use_wrapper)
samples = list(zip(X_test,y_test))

# Convert to the TextAttack-specific Dataset format
attack_dataset = Dataset(samples)
attack_args = AttackArgs(
    num_examples=100, 
    log_to_csv="textbugger_results_second_try_use.csv"
)

attacker = Attacker(use_attack, attack_dataset, attack_args)
results = attacker.attack_dataset()

Second Round of Attacks - Dense Embedding

In [ ]:
embed_attack = TextBuggerLi2018.build(embed_wrapper)
samples = list(zip(X_test,y_test))

# Convert to the TextAttack-specific Dataset format
attack_dataset = Dataset(samples)
attack_args = AttackArgs(
    num_examples=100, 
    log_to_csv="textbugger_results_second_try_embed.csv"
)

attacker = Attacker(embed_attack, attack_dataset, attack_args)
results = attacker.attack_dataset()